# ⚖️ SO SÁNH ĐỐI ĐẦU TOÀN DIỆN: ROI Refiner vs FireGrounder V3 vs YOLO26-Pose

Notebook này thực hiện đánh giá toàn diện, khách quan giữa 3 thế hệ mô hình giải quyết bài toán định vị gốc lửa:
1. **Mô hình A — Two-Stage ROI Refiner (`best_roi.pth`):** MobileNetV4 cắt vùng ảnh vuông (crop 70%) quanh điểm thô để tinh chỉnh tọa độ chân lửa.
2. **Mô hình B — One-Stage Dual-Head FireGrounder V3 (`best_v3_fpn.pth`):** MobileNetV4 + FPN phân loại có/không cháy và dự đoán Heatmap/Regression toàn khung ảnh 256x256.
3. **Mô hình C — One-Stage Single-Keypoint YOLO26-Pose (`best.pt`):** Kiến trúc Pose tiên tiến nhất, đồng thời phát hiện Bounding Box và khóa chặt Keypoint gốc lửa duy nhất trong 1 lần forward.

### Tiêu chí so sánh:
- **Độ chính xác định vị:** MAE (Mean Absolute Error - pixel), Median Error, PCK@5, PCK@10, PCK@25.
- **Tốc độ & Độ nhẹ:** Dung lượng Checkpoint (MB), Số tham số (Parameters), Độ trễ suy luận (ms/ảnh), FPS.
- **Chất lượng hiển thị trực quan:** Các điểm chốt (Check Point) được thể hiện **đậm nét** (không phóng to kích thước, viền đen tương phản cao, đổ bóng chữ rõ ràng).

In [ ]:
# Cell 1: KHỞI TẠO VÀ TẢI CẢ 2 MÔ HÌNH VÀO BỘ NHỚ
import sys, os, time
from pathlib import Path
import torch
import numpy as np
from PIL import Image
from ultralytics import YOLO

# Tự động tìm và thêm LAB_SAM vào sys.path
candidate_lab_paths = [
    Path('LAB_SAM').resolve(),
    Path('../LAB_SAM').resolve(),
    Path('FireGrounder-V3-main/LAB_SAM').resolve()
]
for lp in candidate_lab_paths:
    if lp.exists() and str(lp) not in sys.path:
        sys.path.insert(0, str(lp))

from narrow_localizer import ROIRefinerInference

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'🚀 Thiết bị tính toán: {device}')

# 1. Tải Model A: ROI Refiner (MobileNetV4)
roi_candidates = [
    Path('FireGrounder-V3-main/best_roi.pth').resolve(),
    Path('best_roi.pth').resolve(),
    Path('../FireGrounder-V3-main/best_roi.pth').resolve(),
    Path('LAB_SAM/best_roi.pth').resolve()
]
roi_path = next((p for p in roi_candidates if p.exists()), None)
assert roi_path is not None, 'Không tìm thấy best_roi.pth!'
roi_refiner = ROIRefinerInference(str(roi_path), device=device)
roi_size_mb = roi_path.stat().st_size / 1e6
roi_params = sum(p.numel() for p in roi_refiner.model.parameters())
print(f'✅ Đã tải Model A (ROI Refiner): {roi_path.name} | Dung lượng: {roi_size_mb:.2f} MB | Params: {roi_params/1e6:.2f}M')

# 2. Tải Model B: YOLO26-Pose (One-Stage Single-Keypoint)
yolo_candidates = [
    Path('fire_pose_runs/yolo26n_fire_base/weights/best.pt').resolve(),
    Path('../fire_pose_runs/yolo26n_fire_base/weights/best.pt').resolve(),
    Path('runs/pose/fire_pose_runs/yolo26n_fire_base-2/weights/best.pt').resolve()
]
yolo_path = next((p for p in yolo_candidates if p.exists()), None)
assert yolo_path is not None, 'Không tìm thấy best.pt của YOLO26-Pose!'
yolo_model = YOLO(str(yolo_path))
yolo_size_mb = yolo_path.stat().st_size / 1e6
yolo_params = sum(p.numel() for p in yolo_model.model.parameters())
print(f'✅ Đã tải Model B (YOLO26-Pose): {yolo_path.name} | Dung lượng: {yolo_size_mb:.2f} MB | Params: {yolo_params/1e6:.2f}M')


In [ ]:
# Cell 2: ĐÁNH GIÁ ĐỐI ĐẦU TRỰC TIẾP TRÊN TẬP TEST NGUYÊN BẢN (TEST SET)
# Đo lường sai số khoảng cách Euclidean (Pixel Error) trên cùng một tập ảnh và Ground Truth thật
import sys, os, time, json
from pathlib import Path
import torch
import numpy as np
from PIL import Image
from ultralytics import YOLO

# Tự động import và tải model nếu người dùng chạy thẳng Cell 2
for lp in [Path('LAB_SAM').resolve(), Path('../LAB_SAM').resolve(), Path('FireGrounder-V3-main/LAB_SAM').resolve()]:
    if lp.exists() and str(lp) not in sys.path:
        sys.path.insert(0, str(lp))
from narrow_localizer import ROIRefinerInference

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

if 'roi_refiner' not in globals():
    roi_cand = [Path('FireGrounder-V3-main/best_roi.pth').resolve(), Path('best_roi.pth').resolve(), Path('../FireGrounder-V3-main/best_roi.pth').resolve()]
    r_path = next((p for p in roi_cand if p.exists()), None)
    roi_refiner = ROIRefinerInference(str(r_path), device=device)
    roi_size_mb = r_path.stat().st_size / 1e6
    roi_params = sum(p.numel() for p in roi_refiner.model.parameters())

if 'yolo_model' not in globals():
    yolo_cand = [Path('fire_pose_runs/yolo26n_fire_base/weights/best.pt').resolve(), Path('../fire_pose_runs/yolo26n_fire_base/weights/best.pt').resolve(), Path('runs/pose/fire_pose_runs/yolo26n_fire_base-2/weights/best.pt').resolve()]
    y_path = next((p for p in yolo_cand if p.exists()), None)
    yolo_model = YOLO(str(y_path))
    yolo_size_mb = y_path.stat().st_size / 1e6
    yolo_params = sum(p.numel() for p in yolo_model.model.parameters())

# Tìm file dataset_labels.json linh hoạt
labels_candidates = [
    Path('FireGrounder-V3-main/fire_ground_dataset/dataset_labels.json').resolve(),
    Path('fire_ground_dataset/dataset_labels.json').resolve(),
    Path('../FireGrounder-V3-main/fire_ground_dataset/dataset_labels.json').resolve(),
    Path('../fire_ground_dataset/dataset_labels.json').resolve()
]
labels_path = next((p for p in labels_candidates if p.exists()), None)
assert labels_path is not None, 'Không tìm thấy dataset_labels.json!'

with open(labels_path, 'r', encoding='utf-8') as f:
    all_records = json.load(f)

# Lọc ra các ảnh thuộc test set có lửa
test_records = [r for r in all_records if '/test/' in r['image_path'].replace('\\', '/') and r['has_fire'] == 1]
print(f'Tổng số mẫu kiểm thử có lửa trong Test Set: {len(test_records)}')

# Khởi tạo bộ đo
roi_errors, yolo_errors = [], []
roi_times, yolo_times = [], []

# Đánh giá 100 mẫu đại diện để đo tốc độ và sai số
EVAL_LIMIT = min(100, len(test_records))
print(f'Đang tiến hành chấm điểm đối đầu trên {EVAL_LIMIT} ảnh test...')

for idx in range(EVAL_LIMIT):
    rec = test_records[idx]
    stem = Path(rec['image_path']).name
    
    # Tìm ảnh ở các thư mục test
    img_candidates = [
        Path('dataset_fire_pose/images/test') / stem,
        Path('../dataset_fire_pose/images/test') / stem,
        Path('test/images') / stem,
        Path('../test/images') / stem
    ]
    img_path = next((p for p in img_candidates if p.exists()), None)
    if img_path is None:
        continue
        
    pil_img = Image.open(img_path).convert('RGB')
    W, H = pil_img.size
    gt_x, gt_y = rec['p_fire'][0] * W, rec['p_fire'][1] * H
    
    # ─── ĐO MODEL B: YOLO26-Pose (One-Stage) ───────────────────────────────
    t0 = time.perf_counter()
    res = yolo_model.predict(str(img_path), conf=0.15, imgsz=384, device=device, verbose=False)[0]
    t_yolo = (time.perf_counter() - t0) * 1000
    yolo_times.append(t_yolo)
    
    if len(res.boxes) > 0 and res.keypoints is not None and len(res.keypoints.xy) > 0:
        yolo_pt = res.keypoints.xy[0][0].cpu().numpy()
        y_err = np.sqrt((yolo_pt[0] - gt_x)**2 + (yolo_pt[1] - gt_y)**2)
        yolo_errors.append(y_err)
        box = res.boxes.xyxy[0].cpu().numpy()
        coarse_pt = ((box[0] + box[2])/2, (box[1] + box[3])/2)
    else:
        coarse_pt = (W/2, H/2)
        
    # ─── ĐO MODEL A: ROI Refiner (Two-Stage) ──────────────────────────────
    t0 = time.perf_counter()
    refined = roi_refiner.refine(pil_img, coarse_pt)
    t_roi = (time.perf_counter() - t0) * 1000
    roi_times.append(t_roi)
    r_pt = refined.point
    r_err = np.sqrt((r_pt[0] - gt_x)**2 + (r_pt[1] - gt_y)**2)
    roi_errors.append(r_err)

print(f'🎉 Đã hoàn tất đánh giá đối đầu trên {len(roi_errors)} ảnh!')


In [ ]:
# Cell 3: BẢNG SO SÁNH TỔNG HỢP CÁC CHỈ SỐ KỸ THUẬT QUAN TRỌNG
def compute_metrics(errors, times, model_name, size_mb, params_m):
    errs = np.array(errors)
    return {
        'Dung lượng Checkpoint': f'{size_mb:.2f} MB',
        'Số tham số (M Params)': f'{params_m:.2f}M',
        'MAE Sai số trung bình (Pixel)': f'{np.mean(errs):.2f} px',
        'Median Sai số trung vị (Pixel)': f'{np.median(errs):.2f} px',
        'P90 Sai số percentile 90 (Pixel)': f'{np.percentile(errs, 90):.2f} px',
        'PCK@5px (Tỷ lệ sai số <= 5px)': f'{(errs <= 5).mean()*100:.1f}%',
        'PCK@10px (Tỷ lệ sai số <= 10px)': f'{(errs <= 10).mean()*100:.1f}%',
        'PCK@25px (Tỷ lệ sai số <= 25px)': f'{(errs <= 25).mean()*100:.1f}%',
        'Độ trễ xử lý (Latency)': f'{np.mean(times):.2f} ms',
        'Tốc độ khung hình (FPS)': f'{1000/np.mean(times):.1f} FPS'
    }

row_roi = compute_metrics(roi_errors, roi_times, 'Two-Stage ROI Refiner (MobileNetV4)', roi_size_mb, roi_params/1e6)
row_yolo = compute_metrics(yolo_errors, yolo_times, 'One-Stage YOLO26-Pose (End-to-End)', yolo_size_mb, yolo_params/1e6)

try:
    import pandas as pd
    df_compare = pd.DataFrame([row_roi, row_yolo], index=['Model A: ROI Refiner (MobileNetV4)', 'Model B: YOLO26-Pose (End-to-End)']).T
    display(df_compare)
except Exception:
    print('=' * 85)
    print(f"{'Chỉ số kỹ thuật':<35} | {'Model A: ROI Refiner':<22} | {'Model B: YOLO26-Pose':<22}")
    print('-' * 85)
    for k in row_roi.keys():
        print(f"{k:<35} | {row_roi[k]:<22} | {row_yolo[k]:<22}")
    print('=' * 85)


In [ ]:
# Cell 4: BIỂU ĐỒ PHÂN BỐ SAI SỐ & TỐC ĐỘ
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Boxplot sai số định vị (Pixel)
try:
    axes[0].boxplot([roi_errors, yolo_errors], tick_labels=['ROI Refiner (MobileNetV4)', 'YOLO26-Pose'], patch_artist=True,
                     boxprops=dict(facecolor='lightblue'), medianprops=dict(color='red', linewidth=2))
except TypeError:
    axes[0].boxplot([roi_errors, yolo_errors], labels=['ROI Refiner (MobileNetV4)', 'YOLO26-Pose'], patch_artist=True,
                     boxprops=dict(facecolor='lightblue'), medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Phân bố sai số định vị chân lửa (Pixel Error)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Sai số (Pixel - Càng thấp càng tốt)')
axes[0].grid(True, linestyle='--', alpha=0.6)

# 2. Tốc độ suy luận (ms/ảnh)
models = ['ROI Refiner', 'YOLO26-Pose']
latencies = [np.mean(roi_times), np.mean(yolo_times)]
bars = axes[1].bar(models, latencies, color=['#42A5F5', '#66BB6A'], width=0.4)
axes[1].set_title('Độ trễ suy luận trên GPU RTX 3060 (ms/ảnh)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Độ trễ (ms - Càng thấp càng nhanh)')
for bar in bars:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, f'{yval:.2f} ms\n({1000/yval:.0f} FPS)', ha='center', va='bottom', fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('comparison_roi_vs_yolo26.png', dpi=150)
plt.show()


In [ ]:
# Cell 5: TRỰC QUAN HÓA SO SÁNH TRỰC DIỆN (ĐIỂM CHỐT ĐẬM NÉT - KHÔNG PHÓNG TO KÍCH THƯỚC)
# Hiển thị ảnh kèm: Ground Truth (Xanh lá) vs ROI Refiner (Xanh dương) vs YOLO26-Pose (Đỏ)
import random, cv2

# Hàm vẽ điểm chốt đậm nét (Viền kép đen dày tương phản, tâm màu gốc đặc, chữ có bóng đổ đậm)
def draw_bold_checkpoint(img_rgb, pt, color, label, pos='top_right'):
    x, y = int(pt[0]), int(pt[1])
    h, w = img_rgb.shape[:2]
    x = max(15, min(w - 15, x))
    y = max(15, min(h - 15, y))
    # 1. Viền kép đen dày tương phản cao (giúp điểm nổi bật trên nền sáng lóa hoặc tối)
    cv2.circle(img_rgb, (x, y), 6, (0, 0, 0), 2)
    cv2.circle(img_rgb, (x, y), 5, color, -1)
    cv2.circle(img_rgb, (x, y), 1, (255, 255, 255), -1)
    # 2. Vị trí nhãn thông minh 4 hướng tránh đè chữ nhau
    offsets = {'top_right': (8, -6), 'bottom_right': (8, 16), 'top_left': (-48, -6), 'bottom_left': (-42, 16)}
    dx, dy = offsets.get(pos, (8, 4))
    lx, ly = max(5, min(w - 65, x + dx)), max(15, min(h - 10, y + dy))
    # 3. Viết nhãn cực đậm với viền đen bóng đổ dày
    cv2.putText(img_rgb, label, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(img_rgb, label, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

num_vis = 6
ncols = 2
nrows = (num_vis + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 6 * nrows))
axes = axes.flatten()

random.seed(42)
vis_indices = random.sample(range(len(test_records)), min(num_vis, len(test_records)))

for ax_idx, idx in enumerate(vis_indices):
    rec = test_records[idx]
    stem = Path(rec['image_path']).name
    img_candidates = [
        Path('dataset_fire_pose/images/test') / stem,
        Path('../dataset_fire_pose/images/test') / stem,
        Path('test/images') / stem,
        Path('../test/images') / stem
    ]
    img_path = next((p for p in img_candidates if p.exists()), None)
    if img_path is None:
        continue
        
    img_bgr = cv2.imread(str(img_path))
    H_img, W_img = img_bgr.shape[:2]
    gt_x, gt_y = int(rec['p_fire'][0] * W_img), int(rec['p_fire'][1] * H_img)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # 1. Vẽ Ground Truth (Chấm tròn xanh lá đậm)
    draw_bold_checkpoint(img_rgb, (gt_x, gt_y), (0, 255, 0), 'GT', pos='top_left')
    
    # 2. Vẽ YOLO26-Pose (Chấm đỏ đậm + BBox cam nét thanh)
    res = yolo_model.predict(img_bgr, conf=0.15, imgsz=384, device=device, verbose=False)[0]
    if len(res.boxes) > 0:
        box = res.boxes.xyxy[0].cpu().numpy()
        cv2.rectangle(img_rgb, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (255, 140, 0), 2)
        if res.keypoints is not None and len(res.keypoints.xy) > 0:
            yk = res.keypoints.xy[0][0].cpu().numpy()
            draw_bold_checkpoint(img_rgb, (yk[0], yk[1]), (255, 30, 30), 'YOLO26', pos='bottom_right')
        coarse_pt = ((box[0]+box[2])/2, (box[1]+box[3])/2)
    else:
        coarse_pt = (W_img/2, H_img/2)
        
    # 3. Vẽ ROI Refiner (Chấm xanh dương đậm)
    pil_tmp = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ref = roi_refiner.refine(pil_tmp, coarse_pt)
    draw_bold_checkpoint(img_rgb, (ref.point[0], ref.point[1]), (30, 144, 255), 'ROI', pos='top_right')
    
    axes[ax_idx].imshow(img_rgb)
    axes[ax_idx].set_title(f'{stem}\n[Xanh lá: GT | Xanh dương: ROI Refiner | Đỏ: YOLO26-Pose]', fontsize=11, fontweight='bold')
    axes[ax_idx].axis('off')

plt.tight_layout()
plt.savefig('visual_comparison_roi_vs_yolo26.png', dpi=150)
plt.show()
print('✅ Đã lưu ảnh so sánh trực quan vào: visual_comparison_roi_vs_yolo26.png')


In [ ]:
# Cell 6: SO SÁNH ĐỐI ĐẦU TOÀN DIỆN CẢ 3 MÔ HÌNH: ROI REFINER vs FIREGROUNDER V3 vs YOLO26-POSE
# (Tự động tải 3 mô hình, đo lường định lượng và trực quan hóa với Check Point đậm nét)
import sys, os, time, json, cv2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision.transforms.functional as TF
from ultralytics import YOLO

# 1. Tự động tìm đường dẫn thư mục mã nguồn
for p in [Path('.').resolve(), Path('FireGrounder-V3-main').resolve(), Path('LAB_SAM').resolve(), Path('../LAB_SAM').resolve()]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from firegrounder_v3 import FireGrounderV3, MEAN, STD
from narrow_localizer import ROIRefinerInference

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# 2. Tải cả 3 mô hình vào bộ nhớ (nếu chưa tải)
# Model 1: ROI Refiner
if 'roi_refiner' not in globals():
    r_cand = [Path('FireGrounder-V3-main/best_roi.pth').resolve(), Path('best_roi.pth').resolve(), Path('../FireGrounder-V3-main/best_roi.pth').resolve(), Path('LAB_SAM/best_roi.pth').resolve()]
    r_path = next((p for p in r_cand if p.exists()), None)
    roi_refiner = ROIRefinerInference(str(r_path), device=device)
    roi_size_mb = r_path.stat().st_size / 1e6
    roi_params = sum(p.numel() for p in roi_refiner.model.parameters())
    print(f'✅ Đã tải Model 1 (ROI Refiner): {r_path.name} | {roi_size_mb:.2f} MB')

# Model 2: FireGrounder V3
if 'm_v3' not in globals():
    v3_cand = [Path('FireGrounder-V3-main/v3_outputs/best_v3_fpn.pth').resolve(), Path('v3_outputs/best_v3_fpn.pth').resolve(), Path('../FireGrounder-V3-main/v3_outputs/best_v3_fpn.pth').resolve()]
    v3_path = next((p for p in v3_cand if p.exists()), None)
    m_v3 = FireGrounderV3(pretrained=False).to(device)
    ckpt_v3 = torch.load(str(v3_path), map_location=device, weights_only=False)
    m_v3.load_state_dict(ckpt_v3.get('model_state_dict', ckpt_v3))
    m_v3.eval()
    v3_size_mb = v3_path.stat().st_size / 1e6
    v3_params = sum(p.numel() for p in m_v3.parameters())
    print(f'✅ Đã tải Model 2 (FireGrounder V3): {v3_path.name} | {v3_size_mb:.2f} MB | Epoch {ckpt_v3.get("epoch", "?")}')

# Model 3: YOLO26-Pose
if 'yolo_model' not in globals():
    y_cand = [Path('fire_pose_runs/yolo26n_fire_base/weights/best.pt').resolve(), Path('../fire_pose_runs/yolo26n_fire_base/weights/best.pt').resolve(), Path('runs/pose/fire_pose_runs/yolo26n_fire_base-2/weights/best.pt').resolve()]
    y_path = next((p for p in y_cand if p.exists()), None)
    yolo_model = YOLO(str(y_path))
    yolo_size_mb = y_path.stat().st_size / 1e6
    yolo_params = sum(p.numel() for p in yolo_model.model.parameters())
    print(f'✅ Đã tải Model 3 (YOLO26-Pose): {y_path.name} | {yolo_size_mb:.2f} MB')

# 3. Hàm vẽ điểm chốt đậm nét (Bold Check Point - không tăng kích thước, viền kép tương phản cao, nhãn chữ đổ bóng)
def draw_bold_checkpoint(img_rgb, pt, color, label, pos='top_right'):
    x, y = int(pt[0]), int(pt[1])
    h, w = img_rgb.shape[:2]
    x = max(15, min(w - 15, x))
    y = max(15, min(h - 15, y))
    # 1. Viền kép đen dày tương phản cao (giúp điểm nổi bật trên mọi nền ảnh)
    cv2.circle(img_rgb, (x, y), 6, (0, 0, 0), 2)
    cv2.circle(img_rgb, (x, y), 5, color, -1)
    cv2.circle(img_rgb, (x, y), 1, (255, 255, 255), -1)
    # 2. Vị trí nhãn thông minh 4 hướng tránh đè chữ nhau
    offsets = {'top_right': (8, -6), 'bottom_right': (8, 16), 'top_left': (-48, -6), 'bottom_left': (-42, 16)}
    dx, dy = offsets.get(pos, (8, 4))
    lx, ly = max(5, min(w - 65, x + dx)), max(15, min(h - 10, y + dy))
    # 3. Viết nhãn cực đậm với viền đen bóng đổ dày
    cv2.putText(img_rgb, label, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(img_rgb, label, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

# 4. Đánh giá định lượng trên 50 ảnh Test Set
labels_candidates = [
    Path('FireGrounder-V3-main/fire_ground_dataset/dataset_labels.json').resolve(),
    Path('fire_ground_dataset/dataset_labels.json').resolve(),
    Path('../FireGrounder-V3-main/fire_ground_dataset/dataset_labels.json').resolve(),
    Path('../fire_ground_dataset/dataset_labels.json').resolve()
]
labels_path = next((p for p in labels_candidates if p.exists()), None)
with open(labels_path, 'r', encoding='utf-8') as f:
    all_records = json.load(f)
test_records = [r for r in all_records if '/test/' in r['image_path'].replace('\\', '/') and r['has_fire'] == 1]

roi_errs, v3_errs, yolo_errs = [], [], []
roi_times, v3_times, yolo_times = [], [], []

print(f'⏳ Đang đánh giá đối đầu 3 mô hình trên 50 ảnh test...')
for idx in range(min(50, len(test_records))):
    rec = test_records[idx]
    stem = Path(rec['image_path']).name
    img_candidates = [
        Path('dataset_fire_pose/images/test') / stem,
        Path('../dataset_fire_pose/images/test') / stem,
        Path('test/images') / stem,
        Path('../test/images') / stem
    ]
    img_p = next((p for p in img_candidates if p.exists()), None)
    if img_p is None: continue
    
    pil_img = Image.open(img_p).convert('RGB')
    W, H = pil_img.size
    gt_x, gt_y = rec['p_fire'][0] * W, rec['p_fire'][1] * H
    
    # 1. YOLO26-Pose
    t0 = time.perf_counter()
    res = yolo_model.predict(str(img_p), conf=0.15, imgsz=384, device=device, verbose=False)[0]
    yolo_times.append((time.perf_counter() - t0) * 1000)
    if len(res.boxes) > 0 and res.keypoints is not None and len(res.keypoints.xy) > 0:
        ypt = res.keypoints.xy[0][0].cpu().numpy()
        yolo_errs.append(np.sqrt((ypt[0] - gt_x)**2 + (ypt[1] - gt_y)**2))
        box = res.boxes.xyxy[0].cpu().numpy()
        coarse = ((box[0]+box[2])/2, (box[1]+box[3])/2)
    else:
        coarse = (W/2, H/2)
        
    # 2. ROI Refiner
    t0 = time.perf_counter()
    ref = roi_refiner.refine(pil_img, coarse)
    roi_times.append((time.perf_counter() - t0) * 1000)
    roi_errs.append(np.sqrt((ref.point[0] - gt_x)**2 + (ref.point[1] - gt_y)**2))
    
    # 3. FireGrounder V3
    t0 = time.perf_counter()
    t_v3 = TF.normalize(TF.to_tensor(TF.resize(pil_img, [256, 256])), MEAN, STD).unsqueeze(0).to(device)
    with torch.no_grad():
        out_v3 = m_v3(t_v3)
        if isinstance(out_v3, dict): out_v3 = out_v3['pred']
        pred_v3 = out_v3[0].cpu().numpy()
    v3_times.append((time.perf_counter() - t0) * 1000)
    v3_pt = (pred_v3[1] * W, pred_v3[2] * H)
    v3_errs.append(np.sqrt((v3_pt[0] - gt_x)**2 + (v3_pt[1] - gt_y)**2))

# 5. In bảng so sánh 3 cột
def get_metrics_dict(errs_list, times_list, size_mb, params_m):
    arr = np.array(errs_list)
    return {
        'Dung lượng Checkpoint': f'{size_mb:.2f} MB',
        'Số tham số (Params)': f'{params_m:.2f}M',
        'MAE Sai số TB (Pixel)': f'{np.mean(arr):.2f} px',
        'Median Sai số trung vị': f'{np.median(arr):.2f} px',
        'PCK@10px (Sai số <= 10px)': f'{(arr <= 10).mean()*100:.1f}%',
        'PCK@25px (Sai số <= 25px)': f'{(arr <= 25).mean()*100:.1f}%',
        'Độ trễ xử lý (Latency)': f'{np.mean(times_list):.2f} ms',
        'Tốc độ khung hình (FPS)': f'{1000/np.mean(times_list):.1f} FPS'
    }

metrics_roi = get_metrics_dict(roi_errs, roi_times, roi_size_mb, roi_params/1e6)
metrics_v3 = get_metrics_dict(v3_errs, v3_times, v3_size_mb, v3_params/1e6)
metrics_yolo = get_metrics_dict(yolo_errs, yolo_times, yolo_size_mb, yolo_params/1e6)

try:
    import pandas as pd
    df_3 = pd.DataFrame([metrics_roi, metrics_v3, metrics_yolo], 
                        index=['Model 1: ROI Refiner (MobileNetV4)', 'Model 2: FireGrounder V3 (FPN)', 'Model 3: YOLO26-Pose (One-Stage)']).T
    display(df_3)
except Exception:
    print('=' * 95)
    print(f"{'Chỉ số':<28} | {'1. ROI Refiner':<20} | {'2. FireGrounder V3':<20} | {'3. YOLO26-Pose':<20}")
    print('-' * 95)
    for k in metrics_roi.keys():
        print(f"{k:<28} | {metrics_roi[k]:<20} | {metrics_v3[k]:<20} | {metrics_yolo[k]:<20}")
    print('=' * 95)

# 6. Trực quan hóa 6 ảnh thực tế (2 ảnh mỗi hàng, khổ lớn, Check Point đậm nét)
num_vis = 6
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

vis_idx = 0
for rec in test_records:
    if vis_idx >= num_vis: break
    stem = Path(rec['image_path']).name
    img_candidates = [
        Path('dataset_fire_pose/images/test') / stem,
        Path('../dataset_fire_pose/images/test') / stem,
        Path('test/images') / stem,
        Path('../test/images') / stem
    ]
    img_p = next((p for p in img_candidates if p.exists()), None)
    if img_p is None: continue
    
    img_bgr = cv2.imread(str(img_p))
    H_img, W_img = img_bgr.shape[:2]
    gt_x, gt_y = int(rec['p_fire'][0] * W_img), int(rec['p_fire'][1] * H_img)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # A. Vẽ Ground Truth (Xanh lá đậm)
    draw_bold_checkpoint(img_rgb, (gt_x, gt_y), (0, 255, 0), 'GT', pos='top_left')
    
    # B. Vẽ YOLO26-Pose (Đỏ neon đậm + BBox cam nét thanh)
    res = yolo_model.predict(img_bgr, conf=0.15, imgsz=384, device=device, verbose=False)[0]
    if len(res.boxes) > 0:
        box = res.boxes.xyxy[0].cpu().numpy()
        cv2.rectangle(img_rgb, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (255, 140, 0), 2)
        if res.keypoints is not None and len(res.keypoints.xy) > 0:
            yk = res.keypoints.xy[0][0].cpu().numpy()
            draw_bold_checkpoint(img_rgb, (yk[0], yk[1]), (255, 30, 30), 'YOLO26', pos='bottom_right')
        coarse_pt = ((box[0]+box[2])/2, (box[1]+box[3])/2)
    else:
        coarse_pt = (W_img/2, H_img/2)
        
    # C. Vẽ ROI Refiner (Xanh dương đậm)
    pil_tmp = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ref = roi_refiner.refine(pil_tmp, coarse_pt)
    draw_bold_checkpoint(img_rgb, (ref.point[0], ref.point[1]), (30, 144, 255), 'ROI', pos='top_right')
    
    # D. Vẽ FireGrounder V3 (Cam vàng đậm)
    t_v3 = TF.normalize(TF.to_tensor(TF.resize(pil_tmp, [256, 256])), MEAN, STD).unsqueeze(0).to(device)
    with torch.no_grad():
        out_v3 = m_v3(t_v3)
        if isinstance(out_v3, dict): out_v3 = out_v3['pred']
        pred_v3 = out_v3[0].cpu().numpy()
    v3_px, v3_py = int(pred_v3[1] * W_img), int(pred_v3[2] * H_img)
    draw_bold_checkpoint(img_rgb, (v3_px, v3_py), (255, 180, 0), 'V3', pos='bottom_left')
    
    axes[vis_idx].imshow(img_rgb)
    axes[vis_idx].set_title(f'{stem}\n[Xanh lá: GT | Xanh dương: ROI | Cam: V3 | Đỏ: YOLO26]', fontsize=11, fontweight='bold')
    axes[vis_idx].axis('off')
    vis_idx += 1

plt.tight_layout()
plt.savefig('visual_comparison_3_models.png', dpi=150)
plt.show()
print('✅ Hoàn tất so sánh đối đầu 3 mô hình! Ảnh trực quan đã được lưu vào visual_comparison_3_models.png')
